In [ ]:
# Setup, Version check and Common imports

# Python ≥ 3.7 is required
import sys
assert sys.version_info >= (3, 7)


# TensorFlow ≥ 2.8 is required
import tensorflow as tf
from packaging import version

assert version.parse(tf.__version__) >= version.parse("2.8.0")

# Common imports
import numpy as np
import os
import pandas as pd

from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

print('Python version: ', sys.version_info)
print('TF version: ', tf.__version__)
print('Keras version: ', keras.__version__)
print('GPU is', 'available' if tf.config.list_physical_devices('GPU') else 'NOT AVAILABLE')

**A. Flowers Dataset**

**1. Data Fetching and Loading**

In [ ]:
# Download the Flowers dataset: https://www.kaggle.com/datasets/imsparsh/flowers-dataset

# Create folder flower_photos
# Inside this folder there are 5 subfolders, one for each category

!curl -O https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz

!tar -xzvf flower_photos.tgz

!rm flower_photos.tgz


In [ ]:
# Check the total number of images

import pathlib

os.chdir('flower_photos')
path = os.getcwd()
data_dir = pathlib.Path(path)

image_count = len(list(data_dir.glob('*/*.jpg')))
print('Total images: ', image_count)

In [ ]:
# The images in the original folders are not divided in train and validation datasets
# The following code divides samples into 80% training and 20% validation. No test set is created

# https://www.tensorflow.org/api_docs/python/tf/data/Dataset
# https://www.tensorflow.org/guide/data

# The method image_dataset_from_directory() creates Dataset objects from images located in a specified directory
# https://keras.io/api/data_loading/image/

# Parameters subset and validation_split enable the creation of two datasets (Train: 80%; Validation: 20%)
# This method may shuffle images, adjust size and define the batch size
# This way the dataset is (almost) ready to be processed by the neural network


batch_size = 32
IMG_SIZE = (180, 180)

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="both",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=batch_size,
)

class_names = train_ds.class_names


train_ds = train_ds.cache().prefetch(1)
val_ds = val_ds.cache().prefetch(1)

In [ ]:
# Dataset detailed information

print('Nr. of classes: ', len(class_names))
print('Classes: ', class_names, '\n')

# Cardinality
print('Cardinalidade Treino: ', train_ds.cardinality().numpy())
print('Cardinalidade Validacão: ', val_ds.cardinality().numpy(), '\n')

for image_batch, labels_batch in train_ds:
    print('Tensor Batch Image: ', image_batch.shape)
    print('Tensor Batch Label: ', labels_batch.shape)
    break

In [ ]:
# Visualize a few examples

plt.figure(figsize=(12, 12))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy()/255.)
        plt.title(class_names[labels[i]])
        plt.axis("off")

**2. Creating and Training a Baseline CNN with the Functional API and without Data Augmentation**

In [ ]:
# Creation of a baseline CNN

# It complies with the following constraints:
#   1. Use Keras Functional API
#   2. Use only Conv2D, MaxPooling, Dense, Flatten and Rescaling layers
#   3. Maximum of 5 million parameters
#   4. Without Data Augmentation
#   5. With Input Rescaling

# https://www.tensorflow.org/api_docs/python/tf/keras/Model
# https://www.tensorflow.org/guide/keras/sequential_model
# https://keras.io/api/layers/

keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

# FUNCTIONAL API

inputs = keras.Input(shape=(180,180,3))

a = layers.Rescaling(scale = 1./255)(inputs)

a = layers.Conv2D(filters=32, kernel_size=3, activation='relu', padding='same')(a)
a = layers.MaxPooling2D()(a)

a = layers.Conv2D(32, 3, activation='relu',padding='same')(a)
a = layers.MaxPooling2D()(a)

a = layers.Conv2D(64, 3, activation='relu',padding='same')(a)
a = layers.MaxPooling2D() (a)

a = layers.Flatten()(a)
a = layers.Dense(64, activation='relu')(a)

outputs = layers.Dense(5, activation="softmax")(a)

modelA = keras.Model(inputs=inputs, outputs=outputs)


modelA.summary()


In [ ]:
# Compile and Train the Baseline Model

# Model compilation

#  1. Select the loss function suitable for this situation
#  2. Adopt ADAM optimizer, with default parameterization
#  3. Select accuracy metric to evaluate the model

L = ### COMPLETE ###

Opt = ### COMPLETE ###

modelA.compile(loss = L, optimizer = Opt, metrics=['accuracy'])

# Train for 20 epochs

history = modelA.fit(
  train_ds,
  validation_data=val_ds,
  epochs=20
)

In [ ]:

x = pd.DataFrame(history.history, columns = ['accuracy', 'val_accuracy'])
x.plot(figsize=(8, 5))
plt.grid(True)
plt.show()

**3. Adding Data Augmentation to the Model**


In [ ]:
# Overview on Data Augmentation:
# https://www.tensorflow.org/tutorials/images/data_augmentation
# https://keras.io/api/layers/preprocessing_layers/image_augmentation/


# Visualizing some examples of data augmentation
# The following transformations are applied:
# 1. Random Horizontal Flip
# 2. Random Rotation with range  [-10% * 360, 10% * 360].

example_DA = keras.Sequential([
    layers.RandomFlip(mode='horizontal'),
    layers.RandomRotation(factor=0.1),
])


plt.figure(figsize=(15, 15))

# Take 1 batch from the training dataset (32 images)

# Select one image from the batch (values between 0 and 31)
# Display 9 augmentation from this image

# Change the value to analyze other images from the current batch
im = 1

for image_batch, _ in train_ds.take(1):
    image = image_batch[im]
    for i in range(9):
      ax = plt.subplot(3, 3, i + 1)
      augmented_image = example_DA(image)
      augmented_image = keras.ops.convert_to_numpy(augmented_image)
        # Displays the first image in the output batch. For each of the
        # nine iterations, this is a different augmentation of the same
        # image.
      plt.imshow(augmented_image.astype("uint8"))
      plt.axis("off")






**3.1: Model DA1**

In [ ]:

# Model DA1 relies on Keras preprocessing layers to perform Data Augmentation
# https://keras.io/api/layers/preprocessing_layers/image_augmentation/

keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

# The data augmentation module is created using the Sequential API

DA1 = keras.Sequential([
    layers.RandomFlip(mode='horizontal'),
    layers.RandomRotation(factor=0.1),
])

inputs = keras.Input(shape=(180,180,3))

a = layers.Rescaling(scale = 1./255)(inputs)

a = DA1(a)

a = layers.Conv2D(filters=32, kernel_size=3, activation='relu', padding='same')(a)
a = layers.MaxPooling2D()(a)

a = layers.Conv2D(32, 3, activation='relu',padding='same')(a)
a = layers.MaxPooling2D()(a)

a = layers.Conv2D(64, 3, activation='relu',padding='same')(a)
a = layers.MaxPooling2D() (a)

a = layers.Flatten()(a)
a = layers.Dense(64, activation='relu')(a)

outputs = layers.Dense(5, activation="softmax")(a)

modelDA1 = keras.Model(inputs=inputs, outputs=outputs)


modelDA1.summary()

In [ ]:
# Compile and Train the DA1 Model

# Model compilation

#  1. Select the loss function suitable for this situation
#  2. Adopt ADAM optimizer, with default parameterization
#  3. Select accuracy metric to evaluate the model

L = ### COMPLETE ###

Opt = ### COMPLETE ###

modelDA1.compile(loss = L, optimizer = Opt, metrics=['accuracy'])

# Train for 20 epochs

history = modelDA1.fit(
  train_ds,
  validation_data=val_ds,
  epochs=20
)

In [ ]:
x = pd.DataFrame(history.history, columns = ['accuracy', 'val_accuracy'])
x.plot(figsize=(8, 5))
plt.grid(True)
plt.show()

**3.2: Model DA2**

In [ ]:
# Create a new data augmentation module - Model DA2

# It should comprise 3 or more pre-processing layers, where, at least, two of them,
# must be different from the ones already used

# Add the pre-processing module to the beggining of the NN
# Compile and train the modified CNN and analyze results

# Constraint: 5 million trainable parameters

# You can use additional strategies to fight overfitting


# CODE GOES HERE - MODEL DA2

keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

# The data augmentation module is created using the Sequential API

DA2 = keras.Sequential([

    #### COMPLETE ####

])


inputs = keras.Input(shape=(180,180,3))

a = layers.Rescaling(scale = 1./255)(inputs)

a = DA2(a)

#### COMPLETE #####


outputs = layers.Dense(5, activation="softmax")(a)

modelDA2 = keras.Model(inputs=inputs, outputs=outputs)


modelDA2.summary()


In [ ]:
# Compile and Train the DA2 Model

# Model compilation

#  1. Select the loss function suitable for this situation
#  2. Adopt ADAM optimizer, with default parameterization
#  3. Select accuracy metric to evaluate the model

L = ### COMPLETE ###

Opt = ### COMPLETE ###

modelDA2.compile(loss = L, optimizer = Opt, metrics=['accuracy'])

# Train for 20 epochs

history = modelDA2.fit(
  train_ds,
  validation_data=val_ds,
  epochs=20
)

In [ ]:
x = pd.DataFrame(history.history, columns = ['accuracy', 'val_accuracy'])
x.plot(figsize=(8, 5))
plt.grid(True)
plt.show()